In [4]:
import os
import re
import csv

root_dir = r"listings"
output_csv = "matches.csv"
VERBOSE = True

prefixes = ["svr", "str", "stvr", "oic", "otc", "license", "certificate"]

prefix_regex = re.compile(r"\b(svr|str|stvr|oic|otc|license|certificate)\b", re.IGNORECASE)
number_regex = re.compile(
    r"\b(\d{6}|STR-\d{6}-\d{4}|STR\d{4}-\d{5}|STR\d{4}-\d{4})\b",
    re.IGNORECASE
)

def vprint(msg):
    if VERBOSE:
        print(msg)

def extract_context(line, matched_text):
    words = line.split()
    match_word_index = None

    for i, w in enumerate(words):
        if matched_text in w:
            match_word_index = i
            break

    if match_word_index is not None:
        before = " ".join(words[max(0, match_word_index - 5):match_word_index])
        after = " ".join(words[match_word_index + 1:match_word_index + 6])
    else:
        before = ""
        after = ""

    return before, after

def find_matches_in_file(filepath):
    results = []
    file_has_match = False

    vprint(f"\nScanning file: {filepath}")

    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    lines = lines[6:]  # skip first 6 lines

    cleaned_lines = []
    for line in lines:
        if "----------------------------------------" in line:
            break
        cleaned_lines.append(line)

    for idx, line in enumerate(cleaned_lines, start=7):

        found_prefixes = [p.group() for p in prefix_regex.finditer(line)]
        found_numbers = [n.group() for n in number_regex.finditer(line)]

        # If nothing found, skip
        if not found_prefixes and not found_numbers:
            continue

        file_has_match = True

        # Pair prefixes + numbers if both exist
        if found_prefixes and found_numbers:
            for prefix in found_prefixes:
                for number in found_numbers:
                    matched_text = f"{prefix} {number}"
                    before, after = extract_context(line, number)

                    vprint(f"  PREFIX+NUMBER MATCH: {prefix} → {number}")

                    results.append({
                        "file": filepath,
                        "line_number": idx,
                        "match_type": "prefix_number",
                        "prefix": prefix.upper(),
                        "number": number,
                        "match": matched_text,
                        "before": before,
                        "after": after
                    })

        # Numbers without prefixes
        elif found_numbers:
            for number in found_numbers:
                before, after = extract_context(line, number)

                vprint(f"  NUMBER ONLY MATCH: {number}")

                results.append({
                    "file": filepath,
                    "line_number": idx,
                    "match_type": "number_only",
                    "prefix": "",
                    "number": number,
                    "match": number,
                    "before": before,
                    "after": after
                })

        # Prefix without numbers
        elif found_prefixes:
            for prefix in found_prefixes:
                before, after = extract_context(line, prefix)

                vprint(f"  PREFIX ONLY MATCH: {prefix}")

                results.append({
                    "file": filepath,
                    "line_number": idx,
                    "match_type": "prefix_only",
                    "prefix": prefix.upper(),
                    "number": "",
                    "match": prefix,
                    "before": before,
                    "after": after
                })

    return results, file_has_match


all_results = []
files_with_matches = 0

for root, dirs, files in os.walk(root_dir):
    for file in files:
        if file.lower() == "details.txt":
            full_path = os.path.join(root, file)
            matches, has_match = find_matches_in_file(full_path)
            all_results.extend(matches)
            if has_match:
                files_with_matches += 1

with open(output_csv, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=[
        "file", "line_number", "match_type", "prefix", "number",
        "match", "before", "after"
    ])
    writer.writeheader()
    writer.writerows(all_results)

print(f"\nTotal matches: {len(all_results)}")
print(f"Text files containing matches: {files_with_matches}")



Scanning file: listings\1000743963474861515\details.txt
  PREFIX ONLY MATCH: SVR

Scanning file: listings\1000924249105492404\details.txt
  PREFIX ONLY MATCH: SVR

Scanning file: listings\1001035872868958071\details.txt

Scanning file: listings\1001583781390301814\details.txt

Scanning file: listings\10018689\details.txt

Scanning file: listings\1002837122125630437\details.txt

Scanning file: listings\1002950587602351901\details.txt

Scanning file: listings\1002957922657077094\details.txt

Scanning file: listings\1003075566898464139\details.txt

Scanning file: listings\1003546752038003006\details.txt
  NUMBER ONLY MATCH: STR2024-00002

Scanning file: listings\1003687938507466238\details.txt
  PREFIX ONLY MATCH: license
  PREFIX+NUMBER MATCH: STVR → STR2025-00019

Scanning file: listings\1003756622224095922\details.txt

Scanning file: listings\1003771777824178054\details.txt
  PREFIX ONLY MATCH: STR
  PREFIX ONLY MATCH: Certificate
  PREFIX ONLY MATCH: SVR

Scanning file: listings\1005

In [8]:
import os
import re
import csv

root_dir = r"listings"
output_csv = "matches.csv"
VERBOSE = True

prefix_regex = re.compile(r"\b(svr|str|stvr|oic|otc|license|certificate)\b", re.IGNORECASE)
# Updated to support 5-to-6 digit numbers and hyphenated formats like SVR-02878
number_regex = re.compile(
    r"\b([A-Z]{2,4}-\d{4,6}|\d{5,6}|STR-\d{6}-\d{4}|STR\d{4}-\d{5}|STR\d{4}-\d{4})\b",
    re.IGNORECASE
)

def vprint(msg):
    if VERBOSE:
        print(msg)

def extract_context(line, matched_text):
    words = line.split()
    match_word_index = None

    for i, w in enumerate(words):
        if matched_text in w:
            match_word_index = i
            break

    if match_word_index is not None:
        before = " ".join(words[max(0, match_word_index - 5):match_word_index])
        after = " ".join(words[match_word_index + 1:match_word_index + 6])
    else:
        before = ""
        after = ""

    return before, after

def find_matches_in_file(filepath):
    results = []
    file_has_match = False

    vprint(f"\nScanning file: {filepath}")

    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    lines = lines[6:]  # skip first 6 lines

    cleaned_lines = []
    for line in lines:
        if "----------------------------------------" in line:
            break
        cleaned_lines.append(line)

    seen_matches = set()  # To avoid duplicate entries if caught via adjacent line checks

    for idx, line in enumerate(cleaned_lines, start=7):
        current_line = line.strip()
        # Look ahead to the next line if it exists to catch split labels/values (e.g. "STR Certificate" \n "SVR-02878")
        next_line = cleaned_lines[idx - 7 + 1].strip() if (idx - 6) < len(cleaned_lines) else ""
        combined_text = f"{current_line} {next_line}"

        found_prefixes = [p.group() for p in prefix_regex.finditer(combined_text)]
        found_numbers = [n.group() for n in number_regex.finditer(combined_text)]

        # If nothing found in current/adjacent scope, skip
        if not found_prefixes and not found_numbers:
            continue

        file_has_match = True

        # Pair prefixes + numbers if both exist
        if found_prefixes and found_numbers:
            for prefix in found_prefixes:
                for number in found_numbers:
                    matched_text = f"{prefix} {number}"
                    match_key = (idx, prefix.upper(), number)
                    if match_key in seen_matches:
                        continue
                    seen_matches.add(match_key)

                    before, after = extract_context(combined_text, number)
                    vprint(f"  PREFIX+NUMBER MATCH: {prefix} → {number}")

                    results.append({
                        "file": filepath,
                        "line_number": idx,
                        "match_type": "prefix_number",
                        "prefix": prefix.upper(),
                        "number": number,
                        "match": matched_text,
                        "before": before,
                        "after": after
                    })

        # Numbers without prefixes
        elif found_numbers:
            for number in found_numbers:
                match_key = (idx, "", number)
                if match_key in seen_matches:
                    continue
                seen_matches.add(match_key)

                before, after = extract_context(current_line, number)
                vprint(f"  NUMBER ONLY MATCH: {number}")

                results.append({
                    "file": filepath,
                    "line_number": idx,
                    "match_type": "number_only",
                    "prefix": "",
                    "number": number,
                    "match": number,
                    "before": before,
                    "after": after
                })

        # Prefix without numbers
        elif found_prefixes:
            for prefix in found_prefixes:
                match_key = (idx, prefix.upper(), "")
                if match_key in seen_matches:
                    continue
                seen_matches.add(match_key)

                before, after = extract_context(current_line, prefix)
                vprint(f"  PREFIX ONLY MATCH: {prefix}")

                results.append({
                    "file": filepath,
                    "line_number": idx,
                    "match_type": "prefix_only",
                    "prefix": prefix.upper(),
                    "number": "",
                    "match": prefix,
                    "before": before,
                    "after": after
                })

    return results, file_has_match

all_results = []
files_with_matches = 0

for root, dirs, files in os.walk(root_dir):
    for file in files:
        if file.lower() == "details.txt":
            full_path = os.path.join(root, file)
            matches, has_match = find_matches_in_file(full_path)
            all_results.extend(matches)
            if has_match:
                files_with_matches += 1

with open(output_csv, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=[
        "file", "line_number", "match_type", "prefix", "number",
        "match", "before", "after"
    ])
    writer.writeheader()
    writer.writerows(all_results)

print(f"\nTotal matches: {len(all_results)}")
print(f"Text files containing matches: {files_with_matches}")


Scanning file: listings\1000743963474861515\details.txt
  PREFIX+NUMBER MATCH: SVR → SVR-03101
  PREFIX+NUMBER MATCH: SVR → SVR-03101

Scanning file: listings\1000924249105492404\details.txt
  PREFIX+NUMBER MATCH: SVR → SVR-03102
  PREFIX+NUMBER MATCH: SVR → SVR-03102

Scanning file: listings\1001035872868958071\details.txt

Scanning file: listings\1001583781390301814\details.txt

Scanning file: listings\10018689\details.txt

Scanning file: listings\1002837122125630437\details.txt

Scanning file: listings\1002950587602351901\details.txt

Scanning file: listings\1002957922657077094\details.txt

Scanning file: listings\1003075566898464139\details.txt

Scanning file: listings\1003546752038003006\details.txt
  NUMBER ONLY MATCH: STR2024-00002
  NUMBER ONLY MATCH: STR2024-00002

Scanning file: listings\1003687938507466238\details.txt
  PREFIX ONLY MATCH: license
  PREFIX ONLY MATCH: license
  PREFIX+NUMBER MATCH: STVR → STR2025-00019
  PREFIX+NUMBER MATCH: STVR → STR2025-00019

Scanning fi

In [ ]:
import os
import re
import csv

root_dir = r"listings"
output_csv = "matches.csv"
VERBOSE = True

prefix_regex = re.compile(r"\b(svr|str|stvr|oic|otc|license|certificate)\b", re.IGNORECASE)
number_regex = re.compile(
    r"\b([A-Z]{2,4}-\d{4,6}|\d{5,6}|STR-\d{6}-\d{4}|STR\d{4}-\d{5}|STR\d{4}-\d{4})\b",
    re.IGNORECASE
)

def vprint(msg):
    if VERBOSE:
        print(msg)

def extract_context(line, matched_text):
    words = line.split()
    match_word_index = None

    for i, w in enumerate(words):
        if matched_text in w:
            match_word_index = i
            break

    if match_word_index is not None:
        before = " ".join(words[max(0, match_word_index - 5):match_word_index])
        after = " ".join(words[match_word_index + 1:match_word_index + 6])
    else:
        before = ""
        after = ""

    return before, after

def find_matches_in_file(filepath):
    results = []
    file_has_match = False

    vprint(f"\nScanning file: {filepath}")

    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    lines = lines[6:]  # skip first 6 lines

    cleaned_lines = []
    for line in lines:
        if "----------------------------------------" in line:
            break
        cleaned_lines.append(line)

    seen_line_numbers = set()  # Tracks (line_number, number) to prevent duplicate prefix rows

    for idx, line in enumerate(cleaned_lines, start=7):
        current_line = line.strip()
        next_line = cleaned_lines[idx - 7 + 1].strip() if (idx - 6) < len(cleaned_lines) else ""
        combined_text = f"{current_line} {next_line}"

        found_prefixes = [p.group() for p in prefix_regex.finditer(combined_text)]
        found_numbers = [n.group() for n in number_regex.finditer(combined_text)]

        # Skip if nothing found or if there is no number in the line/context (drops prefix_only)
        if not found_prefixes and not found_numbers:
            continue
        if not found_numbers:
            continue

        file_has_match = True

        # Pair prefixes + numbers if prefixes exist
        if found_prefixes:
            for number in found_numbers:
                unique_key = (idx, number)
                if unique_key in seen_line_numbers:
                    continue
                seen_line_numbers.add(unique_key)

                # Pick the best prefix: prefer one matching the number's alpha prefix (e.g., SVR for SVR-02878)
                best_prefix = found_prefixes[0]
                num_upper = number.upper()
                for p in found_prefixes:
                    if num_upper.startswith(p.upper() + "-") or num_upper.startswith(p.upper()):
                        best_prefix = p
                        break

                matched_text = f"{best_prefix} {number}"
                before, after = extract_context(combined_text, number)

                vprint(f"  PREFIX+NUMBER MATCH: {best_prefix} → {number}")

                results.append({
                    "file": filepath,
                    "line_number": idx,
                    "match_type": "prefix_number",
                    "prefix": best_prefix.upper(),
                    "number": number,
                    "match": matched_text,
                    "before": before,
                    "after": after
                })
        else:
            # Numbers without prefixes
            for number in found_numbers:
                unique_key = (idx, number)
                if unique_key in seen_line_numbers:
                    continue
                seen_line_numbers.add(unique_key)

                before, after = extract_context(current_line, number)
                vprint(f"  NUMBER ONLY MATCH: {number}")

                results.append({
                    "file": filepath,
                    "line_number": idx,
                    "match_type": "number_only",
                    "prefix": "",
                    "number": number,
                    "match": number,
                    "before": before,
                    "after": after
                })

    return results, file_has_match

all_results = []
files_with_matches = 0

for root, dirs, files in os.walk(root_dir):
    for file in files:
        if file.lower() == "details.txt":
            full_path = os.path.join(root, file)
            matches, has_match = find_matches_in_file(full_path)
            all_results.extend(matches)
            if has_match:
                files_with_matches += 1

with open(output_csv, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=[
        "file", "line_number", "match_type", "prefix", "number",
        "match", "before", "after"
    ])
    writer.writeheader()
    writer.writerows(all_results)

print(f"\nTotal matches: {len(all_results)}")
print(f"Text files containing matches: {files_with_matches}")


Scanning file: listings\1000743963474861515\details.txt
  PREFIX+NUMBER MATCH: SVR → SVR-03101
  PREFIX+NUMBER MATCH: SVR → SVR-03101

Scanning file: listings\1000924249105492404\details.txt
  PREFIX+NUMBER MATCH: SVR → SVR-03102
  PREFIX+NUMBER MATCH: SVR → SVR-03102

Scanning file: listings\1001035872868958071\details.txt

Scanning file: listings\1001583781390301814\details.txt

Scanning file: listings\10018689\details.txt

Scanning file: listings\1002837122125630437\details.txt

Scanning file: listings\1002950587602351901\details.txt

Scanning file: listings\1002957922657077094\details.txt

Scanning file: listings\1003075566898464139\details.txt

Scanning file: listings\1003546752038003006\details.txt
  NUMBER ONLY MATCH: STR2024-00002
  NUMBER ONLY MATCH: STR2024-00002

Scanning file: listings\1003687938507466238\details.txt
  PREFIX+NUMBER MATCH: STVR → STR2025-00019
  PREFIX+NUMBER MATCH: STVR → STR2025-00019

Scanning file: listings\1003756622224095922\details.txt

Scanning fil

: 